# Encoding as a sharpness dial

Notebook 05 argued the residual over-doubling is the *smoothness* of the representation: a continuous
approximator interpolates across adjacent cells, so a hard boundary becomes a soft ramp and the cells
on the ramp flip. That's a **testable** claim — if it's smoothness, then making the input *less*
smooth should sharpen the boundary and shrink the over-doubles.

We have the dial: the **encoding**. `scalar` feeds player total and dealer upcard as single ordered
numbers (smooth, ordinal); `onehot` feeds each as a categorical block with no ordering between values.
Same network, same schedule, same everything — only the encoding differs. This notebook compares them
and finds: one-hot **halves the over-doubles** (confirming the cause), the two encodings invest their
representation in opposite things (ordinal *axes* vs categorical *walls*), and — because the residual
here is a boundary problem — the sharper encoding wins.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
sys.path.insert(0, '.')
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'blackjack_rl').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from blackjack_rl.analysis_loader import load_runs
from blackjack_rl.dqn.embedding import load_agent, cell_embeddings
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

df = load_runs(); dqn = df[df.has_model & (df.method == 'dqn')]
# the clean A/B: one-hot vs scalar, identical flat-then-decay config, encoding the only difference.
def pick(encoding, episodes=1_500_000):
    g = dqn[(dqn.encoding == encoding) & (dqn.lr_hold_until > 0) & (dqn.episodes == episodes)]
    assert len(g), f'no run with model for {encoding} {episodes}'
    return g.iloc[0]
RUN_A = pick('onehot')   # categorical / sharp
RUN_B = pick('scalar')   # ordinal / smooth
for tag, R in [('A onehot', RUN_A), ('B scalar', RUN_B)]:
    print(f'{tag}: {R.run}  agree {R.agreement:.1%}  edge {R.edge_pct:.2f}%   '
          f'(sched {R.lr_schedule} hold {R.lr_hold_until}, batch {R.batch}, {R.episodes//1000}k, seed {R.seed})')

In [ ]:
def diff_cells(path):
    return json.load(open(Path(path) / 'record.json', encoding='utf-8'))['diff']['cells']
def breakdown(path):
    dis = [c for c in diff_cells(path) if c['category'] == 'genuine_disagreement']
    over = sum(1 for c in dis if c['agent_action'] == 'double' and c['basic_action'] != 'double')
    under = sum(1 for c in dis if c['basic_action'] == 'double' and c['agent_action'] != 'double')
    return dict(genuine=len(dis), over=over, under=under)
def embed(row):
    ce = cell_embeddings(load_agent(row.path)); X = np.array(ce.embeddings)
    p = PCA(n_components=2).fit(X)
    return dict(label=f"{row.encoding} ({row.run[:15]})", cells=ce.cells, X=X,
                pca=p.transform(X), tsne=TSNE(n_components=2, perplexity=30, init='pca', random_state=0).fit_transform(X))
A = embed(RUN_A); B = embed(RUN_B)

ACT  = {'hit': '#378ADD', 'stand': '#1D9E75', 'double': '#D85A30'}
CAT  = {'agree': '#CBCBC4', 'near_equal_ev': '#EF9F27', 'genuine_disagreement': '#E24B4A'}
SOFT = {'soft': '#7F77DD', 'hard': '#9AA0A6'}
KEY = {'action': lambda c: c['action'], 'category': lambda c: c['category'], 'soft': lambda c: 'soft' if c['is_soft'] else 'hard'}
PAL = {'action': ACT, 'category': CAT, 'soft': SOFT}
CONT = {'upcard': ('dealer_upcard', 'coolwarm')}
def scat(ax, pts, labels, colors, title):
    for lab in dict.fromkeys(labels):
        m = [i for i, l in enumerate(labels) if l == lab]
        ax.scatter(pts[m, 0], pts[m, 1], s=22, c=colors.get(lab, '#888'), label=str(lab), edgecolor='none', alpha=0.85)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([]); ax.legend(fontsize=8, framealpha=0.9)
def compare(kind, proj='pca'):
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    for j, R in enumerate([A, B]):
        P = R[proj]
        if kind in CONT:
            field, cmap = CONT[kind]
            sc = ax[j].scatter(P[:, 0], P[:, 1], s=22, c=[c[field] for c in R['cells']], cmap=cmap)
            ax[j].set_title(f"{R['label']} — {kind}"); ax[j].set_xticks([]); ax[j].set_yticks([])
            fig.colorbar(sc, ax=ax[j], fraction=0.046, pad=0.04)
        else:
            scat(ax[j], P, [KEY[kind](c) for c in R['cells']], PAL[kind], f"{R['label']} — {kind}")
    fig.suptitle(f"{proj.upper()}  ·  {kind}    (A one-hot | B scalar)", y=1.03); plt.tight_layout(); plt.show()
def sil(R, key):
    labs = [key(c) for c in R['cells']]; code = {a: i for i, a in enumerate(dict.fromkeys(labs))}
    return silhouette_score(R['X'], [code[l] for l in labs])

## Part 1 — scalar renders the ordinal axes as clean gradients

Three of the inputs are *ordered*: player total, the soft flag, and the dealer upcard. Scalar feeds
them as numbers, so the network can lay cells along a smooth axis and interpolate; one-hot feeds them
as unordered categories, so there is no axis to form a gradient on. The soft/hard and upcard panels
show it directly: in **scalar (B)** soft separates cleanly and the upcard runs as a smooth
weak→strong gradient; in **one-hot (A)** both are scrambled. The silhouettes put a number on it.

In [ ]:
compare('soft')
compare('upcard')
print(f"{'':14}{'soft/hard sil':>16}{'upcard sil':>14}")
for R in [A, B]:
    print(f"{R['label'][:13]:14}{sil(R, KEY['soft']):>16.3f}{sil(R, lambda c: c['dealer_upcard']):>14.3f}")
print('\nhigher = the ordered feature is laid out as clean structure. scalar should lead both.')

## Part 2 — one-hot draws a sharper boundary, and the over-doubles halve

The flip side, and the direct test of 05's claim. Because one-hot treats each total/upcard as its own
slot, it can place a *hard* edge between adjacent cells instead of ramping across them — so the double
region is more cleanly bounded and the boundary over-doubles drop. The action and category panels show
the sharper double block in **one-hot (A)** and the stringy, error-lined double in **scalar (B)**; the
counts confirm it — and the price one-hot pays is a couple of *under*-doubles, the doubles it can no
longer infer from a neighbouring cell.

In [ ]:
compare('action')
compare('category')
print(f"{'encoding':10}{'genuine':>9}{'over-dbl':>10}{'under-dbl':>11}")
for tag, R in [('one-hot', RUN_A), ('scalar', RUN_B)]:
    b = breakdown(R.path)
    print(f"{tag:10}{b['genuine']:>9}{b['over']:>10}{b['under']:>11}")
print('\none-hot roughly halves the over-doubles (sharper boundary) and adds a couple of under-doubles\n'
      '(no interpolation to carry a double in from a neighbour) — confirming the cause is smoothness.')

## Part 3 — the net effect, and the crossover

So which wins? It depends on what the residual error *is*. For Problem A the residual is over-doubling
— a boundary problem — so one-hot's sharper boundary wins on **both** agreement and edge here; scalar's
ordinal generalization (cleaner structure, a few doubles inferred from neighbours) isn't enough to
offset a dozen extra over-doubles. But the *trajectory* shows the two strengths taking turns: scalar
leads while only the ordinal hit/stand decision exists, and one-hot overtakes the moment the
sharp-boundary double is introduced. The crossover is the tradeoff in motion.

In [ ]:
print(f"{'encoding':10}{'agreement':>11}{'edge':>8}{'over-dbl':>10}")
for tag, R in [('one-hot', RUN_A), ('scalar', RUN_B)]:
    print(f"{tag:10}{R.agreement:>11.1%}{R.edge_pct:>7.2f}%{breakdown(R.path)['over']:>10}")
print('-> here one-hot leads BOTH metrics: the residual was boundary over-doubling, exactly its strength.')
print()
def agreement_curve(path):
    lc = json.load(open(Path(path) / 'record.json', encoding='utf-8'))['learning_curve']
    return [(cp['episode'], cp['agreement']) for cp in lc if cp.get('agreement') is not None]
fig, ax = plt.subplots(figsize=(9, 4.5))
for tag, R, col in [('one-hot (sharp boundary)', RUN_A, '#D85A30'), ('scalar (ordinal)', RUN_B, '#378ADD')]:
    c = agreement_curve(R.path)
    ax.plot([e for e, _ in c], [a for _, a in c], color=col, lw=1.4, label=tag)
ax.axvline(RUN_A.lr_hold_until, ls='--', color='#999', lw=1)
ax.text(RUN_A.lr_hold_until + 8000, 0.71, 'double introduced', fontsize=8, color='#666')
ax.set_xlabel('training episode'); ax.set_ylabel('agreement'); ax.legend(fontsize=8, loc='lower right')
ax.set_title('The crossover: scalar leads the hit/stand phase, one-hot overtakes once double enters')
plt.tight_layout(); plt.show()

## Conclusion — smoothness helps structure, hurts boundaries

The encoding is a single dial between two regimes, and the experiment lands cleanly on both ends:

- **scalar = smooth / ordinal.** It renders the ordered inputs (total, soft/hard, dealer upcard) as
  clean gradients and interpolates across neighbours — strong global structure and rare-cell
  generalization (visible as its early-training lead on the hit/stand phase). The cost is a soft
  double boundary, so it over-doubles at the seam.
- **one-hot = categorical / sharp.** It draws hard walls between adjacent cells — halving the
  over-doubles — at the cost of the ordinal structure (scrambled soft/hard and upcard, a few
  under-doubles, weaker generalization).

Which wins is decided by the residual. For Problem A the residual is boundary over-doubling, so turning
the dial toward *sharp* (one-hot) wins both agreement and edge here; scalar's generalization would pull
ahead only where rare-cell coverage, not boundaries, is the bottleneck. Either way, the experiment
confirms 05's diagnosis directly — the over-doubling *was* smoothness, and turning the dial removes
half of it.

And it sharpens notebook 01's verdict. The table wins on **agreement-exactness** because it is the
ultimate sharp boundary — no "between" at all, zero tuning. But on **edge**, the metric that actually
costs money, a one-hot net draws boundaries sharp enough to *match or beat* the table (the 2.5M one-hot
run reaches ~0.87% edge, below the no-split table's ~1.3% — single-seed, worth confirming). So the
honest one-line verdict for Problem A: **the table wins on exactness and zero-tuning, not on money —
and the gap that remains is a representation choice (smooth vs sharp), not a limit on what the network
can learn.**